[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_11_advantage_actor_critic.ipynb)

<div style="text-align:center">
    <h1>
        Advantage Actor-Critic (A2C)
    </h1>
</div>

<br><br>

<div style="text-align:center">
In this notebook we are going to combine temporal difference learning (TD) with policy gradient methods. The resulting algorithm is called Advantage Actor-Critic (A2C) and uses a one-step estimate of the return to update the policy:
</div>

\begin{equation}
\hat G_t = R_{t+1} + \gamma v(S_{t+1}|w)
\end{equation}


<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_policy_network, plot_stats, seed_everything


## Import the necessary software libraries:

In [ ]:
import os
import torch
import gym
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch import nn as nn
from torch.optim import AdamW
import torch.nn.functional as F

## Create and preprocess the environment

### Create the environment

In [ ]:
env = gym.make('Acrobot-v1')

In [ ]:
dims = env.observation_space.shape[0]
actions = env.action_space.n

print(f"State dimensions: {dims}. Actions: {actions}")
print(f"Sample state: {env.reset()}")

In [ ]:
plt.imshow(env.render(mode='rgb_array'))

### Prepare the environment to work with PyTorch

In [ ]:
class PreprocessEnv(gym.Wrapper):

    def __init__(self, env):
        gym.Wrapper.__init__(self, env)

    def reset(self):
        state = self.env.reset()
        return torch.from_numpy(state).float()

    def step(self, actions):
        actions = actions.squeeze().numpy()
        next_state, reward, done, info = self.env.step(actions)
        next_state = torch.from_numpy(next_state).float()
        reward = torch.tensor(reward).unsqueeze(1).float()
        done = torch.tensor(done).unsqueeze(1)
        return next_state, reward, done, info

In [ ]:
num_envs = 8
parallel_env = gym.vector.make('Acrobot-v1', num_envs=num_envs)
seed_everything(parallel_env)
parallel_env = PreprocessEnv(parallel_env)

### Create the policy $\pi(s)$ -- the **Actor**

**Your turn.** Build the *actor* network in the cell below.

The **actor** is the policy: given a state it outputs a **probability for each action**. It is the part that decides what to do. Use a couple of hidden layers with non-linear activations and finish with a layer that has one output per action followed by a **Softmax** so the outputs are probabilities.

### Create the value network $v(s)$ -- the **Critic**

**Your turn.** Build the *critic* network in the cell below.

The **critic** estimates how good a state is: given a state it outputs a **single number** $v(s)$. It does not choose actions; it provides a **baseline** the actor is measured against. The architecture looks like the actor but ends in a single output and has **no Softmax** (it is a value, not a probability).

## Implement the algorithm

### Advantage Actor-Critic -- the idea you are about to code

A2C trains **two networks together**:

- the **actor** (policy) decides what to do,
- the **critic** (value function $v(s)$) judges how good a state is and acts as a **baseline**.

Instead of waiting for the whole episode like REINFORCE, A2C uses a **one-step (TD) estimate** of the return,
$$\hat G_t = R_{t+1} + \gamma\, v(S_{t+1}),$$
and from it the **advantage**
$$A_t = \big(R_{t+1} + \gamma\, v(S_{t+1})\big) - v(S_t).$$
Positive advantage means the action beat expectations -> make it more likely; negative means make it less likely.

**Your turn.** Write the `actor_critic` training loop in the cell below. At each step you do two updates:
1. **Critic update** -- move $v(S_t)$ toward the TD target $\hat G_t$ with an MSE loss (remember to `detach()` the target and to zero the future value when the episode ended).
2. **Actor update** -- use the (detached) **advantage** to weight the log-probability of the action taken; optionally add a small **entropy** bonus.

Give the actor and critic their own optimisers, and record the losses and the episode return for plotting.

**Finally:** call your `actor_critic` function to train both networks and save the result in a variable named `stats` so the plotting cell below works.

## Show results

### Show execution stats

In [ ]:
plot_stats(stats)

### Test the resulting agent

In [ ]:
test_policy_network(env, actor, episodes=2)

## Resources

[[1] Reinforcement Learning: An Introduction. Ch.13](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)